# Laboratorio 7 — NLP end-to-end y Embeddings

## Datos, preprocesamiento y Skip-Gram with Negative Sampling

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
import math, random, re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch import nn
from torch.utils.data import DataLoader, IterableDataset

SEED = 42
TOKEN_LIMIT = 20_000_000
MIN_COUNT = 5
WINDOW_SIZE = 5
SUBSAMPLE_THRESHOLD = 1e-5
PAD, UNK = '<pad>', '<unk>'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TOKEN_RE = re.compile(r"[a-z]+(?:['-][a-z]+)*|\d+(?:[.,]\d+)*", re.I)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

### 1. Normalización y tokenización

La decisión inicial es convertir a minúsculas, descartar puntuación aislada y conservar contracciones y guiones internos. Los números se mantienen textuales para no introducir `<num>`, token que no necesariamente existe en GloVe. Esta misma función deberá aplicarse a AG News.

In [ ]:
def tokenize(text, lowercase=True, replace_numbers=False):
    if lowercase:
        text = text.lower()
    tokens = TOKEN_RE.findall(text)
    return ['<num>' if replace_numbers and token[0].isdigit() else token for token in tokens]

examples = [
    "I can't believe it's already Jan. 5th!",
    "The state-of-the-art model costs $1,299.99.",
    "Dr. Smith lives in the U.S.A.",
    "We're testing e-mail and co-operate.",
    "Don't split O'Connor incorrectly.",
    "The 3.14 value isn't an integer.",
    "NASA's launch was re-scheduled.",
    "Mr. Jones said: 'Hello!'",
    "This is a well-known edge-case.",
    "They'll arrive at 10:30 p.m.",
]
import spacy
nlp = spacy.blank('en')
comparison = pd.DataFrame({
    'oracion': examples,
    'tokenizador_propio': [tokenize(text) for text in examples],
    'spacy': [[token.text for token in nlp(text)] for text in examples],
})
comparison['coinciden'] = comparison.tokenizador_propio == comparison.spacy
comparison

In [ ]:
def build_corpus(split, limit=TOKEN_LIMIT):
    documents, articles, lines, tokens = [], 0, 0, 0
    for row in split:
        articles += 1
        words = tokenize(row['text'])
        if not words:
            continue
        remaining = limit - tokens
        if remaining <= 0:
            break
        words = words[:remaining]
        documents.append(words)  # no cruzar contextos entre líneas
        lines += 1; tokens += len(words)
        if tokens == limit:
            break
    return documents, {'articulos': articles, 'lineas_no_vacias': lines, 'tokens': tokens}

wikitext_train = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train')
documents, corpus_stats = build_corpus(wikitext_train)
corpus_stats

In [ ]:
# Frecuencias, ley de Zipf y cobertura de palabras frecuentes
counts = Counter(token for document in documents for token in document)
total_tokens = sum(counts.values())
ranked = counts.most_common()
coverage = {k: sum(count for _, count in ranked[:k]) / total_tokens for k in (10, 1_000, 30_000)}
print('Cobertura:', coverage)
plt.figure(figsize=(7, 4))
plt.loglog(range(1, len(ranked) + 1), [count for _, count in ranked])
plt.xlabel('Rango'); plt.ylabel('Frecuencia'); plt.title('Ley de Zipf en WikiText')
plt.show()

def make_vocab(min_count):
    tokens = [token for token, count in ranked if count >= min_count]
    id_to_token = [PAD, UNK, *tokens]
    token_to_id = {token: index for index, token in enumerate(id_to_token)}
    unknown = sum(count for token, count in counts.items() if token not in token_to_id) / total_tokens
    return token_to_id, id_to_token, unknown

vocab_sweep = []
for threshold in (2, 5, 10):
    mapping, terms, unknown_rate = make_vocab(threshold)
    vocab_sweep.append({'min_count': threshold, 'tamano_vocabulario': len(terms), 'porcentaje_desconocido': 100 * unknown_rate})
pd.DataFrame(vocab_sweep)

In [ ]:
# Subsampling y pares (central, contexto) para skip-gram
token_to_id, id_to_token, unknown_rate = make_vocab(MIN_COUNT)

def keep_probability(token):
    frequency = counts[token] / total_tokens
    return min(1.0, (math.sqrt(frequency / SUBSAMPLE_THRESHOLD) + 1) * SUBSAMPLE_THRESHOLD / frequency)

rng = random.Random(SEED)
sampled_documents = [[token for token in document if rng.random() < keep_probability(token)] for document in documents]
sampled_documents = [document for document in sampled_documents if document]
discarded = {word: 100 * (1 - keep_probability(word)) for word in ('the', 'of', 'and')}

def encode(documents):
    return [[token_to_id.get(token, token_to_id[UNK]) for token in document] for document in documents]

def pair_count(encoded_documents, window=WINDOW_SIZE):
    return sum(sum(min(i, window) + min(len(document) - i - 1, window) for i in range(len(document))) for document in encoded_documents)

encoded = encode(documents)
sampled_encoded = encode(sampled_documents)
pd.Series({'vocabulario': len(id_to_token), 'oov_corpus_pct': 100 * unknown_rate, 'pares_antes': pair_count(encoded), 'pares_despues': pair_count(sampled_encoded), **{f'descartado_{word}_pct': rate for word, rate in discarded.items()}})

### 2. SGNS propio en PyTorch

Se usan dos tablas: `input_embeddings` es W y `output_embeddings` es W'. Al finalizar se exporta W como representación de palabras.

In [ ]:
def skipgram_pairs(encoded_documents, window=WINDOW_SIZE):
    for document in encoded_documents:
        for i, center in enumerate(document):
            for j in range(max(0, i-window), min(len(document), i+window+1)):
                if i != j:
                    yield center, document[j]

class PairDataset(IterableDataset):
    def __init__(self, docs, window): self.docs, self.window = docs, window
    def __iter__(self): yield from skipgram_pairs(self.docs, self.window)

class SGNS(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        nn.init.uniform_(self.input_embeddings.weight, -0.5/embedding_dim, 0.5/embedding_dim)
        nn.init.zeros_(self.output_embeddings.weight)
    def forward(self, centers, positives, negatives):
        center = self.input_embeddings(centers)
        positive = (center * self.output_embeddings(positives)).sum(1)
        negative = torch.bmm(self.output_embeddings(negatives), center.unsqueeze(2)).squeeze(2)
        return -(torch.nn.functional.logsigmoid(positive) + torch.nn.functional.logsigmoid(-negative).sum(1)).mean()

negative_weights = torch.tensor([counts.get(token, 0)**0.75 for token in id_to_token], dtype=torch.float)
negative_weights[:2] = 0
negative_weights /= negative_weights.sum()

def train_sgns(model, docs, epochs=5, negatives=10, batch_size=2048, learning_rate=0.025, metric_callback=None):
    model.to(DEVICE); optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    for epoch in range(1, epochs + 1):
        if DEVICE.type == 'cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
        started = perf_counter(); loss_sum = 0; pairs = 0
        for centers, positives in DataLoader(PairDataset(docs, WINDOW_SIZE), batch_size=batch_size):
            centers, positives = centers.to(DEVICE), positives.to(DEVICE)
            negative_ids = torch.multinomial(negative_weights.to(DEVICE), centers.numel() * negatives, replacement=True).view(-1, negatives)
            optimizer.zero_grad(set_to_none=True)
            loss = model(centers, positives, negative_ids); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(centers); pairs += len(centers)
        metrics = metric_callback(model, epoch) if metric_callback else {}
        peak = torch.cuda.max_memory_allocated(DEVICE) / 2**20 if DEVICE.type == 'cuda' else np.nan
        history.append({'epoch': epoch, 'loss': loss_sum/pairs, 'pares': pairs, 'segundos': perf_counter()-started, 'gpu_mib': peak, **metrics})
    return pd.DataFrame(history)

In [ ]:
# Corrida base 100D. Repetir cambiando dimensión (50, 100, 300) y fracción del corpus (25, 50, 100 %).
model = SGNS(len(id_to_token), embedding_dim=100)
history = train_sgns(model, sampled_encoded, epochs=5, negatives=10, batch_size=2048, learning_rate=0.025)
history.plot(x='epoch', y='loss', marker='o', title='Pérdida SGNS por epoch')
plt.show()

# Exportar W en formato word2vec.
output = Path('../artifacts/sgns_best_vectors.txt')
output.parent.mkdir(exist_ok=True)
vectors = model.input_embeddings.weight.detach().cpu().numpy()
with output.open('w', encoding='utf-8') as file:
    file.write(f'{len(id_to_token)-2} {vectors.shape[1]}\n')
    for token, vector in zip(id_to_token[2:], vectors[2:]):
        file.write(f"{token} {' '.join(map(str, vector))}\n")
print(f'Vectores guardados en {output}')